# pipeline and splits: Grouped split and baseline pipeline
The split was generated by `scripts/make_splits.py`. Its aggregate test row and class counts are shown for balance only; no test metrics are computed.

In [1]:
from pathlib import Path
import pandas as pd

root = Path.cwd() if (Path.cwd() / 'AGENTS.md').exists() else Path.cwd().parent
tables = root / 'reports' / 'tables'
print(pd.read_csv(tables / 'split_summary.csv').to_string(index=False))


split  rows  positives  prevalence
 test  3999        508    0.127032
train 18657       2368    0.126923
  val  3997        507    0.126845


## Training-only leakage tripwire
Each raw feature was evaluated alone using grouped five-fold average precision on training rows. Values above three times training prevalence would stop the scan.

In [2]:
print(pd.read_csv(tables / 'single_feature_scan.csv').sort_values('mean_pr_auc', ascending=False).to_string(index=False))


                feature  mean_pr_auc  std_pr_auc  train_prevalence  flag_over_3x
        education_level     0.167650    0.006419          0.126923         False
         head_education     0.157569    0.007478          0.126923         False
  census_household_size     0.143784    0.004190          0.126923         False
               can_read     0.141396    0.003791          0.126923         False
              can_write     0.139757    0.003804          0.126923         False
     currently_enrolled     0.139034    0.001130          0.126923         False
                    age     0.138820    0.002849          0.126923         False
         marital_status     0.138115    0.001180          0.126923         False
   relationship_to_head     0.137712    0.001636          0.126923         False
           has_training     0.137624    0.001618          0.126923         False
               province     0.137131    0.003636          0.126923         False
census_urban_proportion     

## Validation baselines
`scripts/compute_baselines.py` fits each pipeline on training rows and evaluates only validation rows. PR-AUC should be read alongside prevalence; accuracy is included for completeness.

In [3]:
print(pd.read_csv(tables / 'baseline_validation.csv').to_string(index=False))


        model       f1  precision   recall   pr_auc  roc_auc  accuracy  prevalence  lift_at_10_percent   tn  fp  fn  tp
   stratified 0.123506   0.124748 0.122288 0.126589 0.498823  0.779835    0.126845            0.965745 3055 435 445  62
most_frequent 0.000000   0.000000 0.000000 0.126845 0.500000  0.873155    0.126845            1.083999 3490   0 507   0


## Preprocessing
The single pipeline learns the household-size 99th percentile, imputation values, scale and one-hot categories from training rows. Model tuning, calibration and final test evaluation remain for later stages.